# Dictionary Investigation

In [ ]:
import csv
from collections import defaultdict

# Path to the TSV file
file_path = r"MPCD\dictionary exports\mpcd_dictionary-jan2026.tsv"

total_entries = 0
word_count_stats = defaultdict(int)
lemmata_by_length = defaultdict(list)

with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    
    for row in reader:
        lemma = row["word"].strip()
        
        # Skip empty lemmata if any
        if not lemma:
            continue
        
        total_entries += 1
        
        # Count words using space as boundary
        word_count = len(lemma.split())
        
        word_count_stats[word_count] += 1
        lemmata_by_length[word_count].append(lemma)

# Results
print(f"Total entries: {total_entries}")
print(f"One-word lemmata: {word_count_stats[1]}")
print(f"Two-word lemmata: {word_count_stats[2]}")
print(f"Three-word lemmata: {word_count_stats[3]}")
print(f"Four-word lemmata: {word_count_stats[4]}")

# Optional: check if there are longer lemmata
longer = {k: v for k, v in word_count_stats.items() if k > 4}
if longer:
    print("\nLemmata longer than four words:")
    for length, count in sorted(longer.items()):
        print(f"{length} words: {count}")


Total entries: 19335
One-word lemmata: 18012
Two-word lemmata: 774
Three-word lemmata: 443
Four-word lemmata: 87

Lemmata longer than four words:
5 words: 15
6 words: 4


In [2]:
# print examples
print("\nTwo-word lemmata:")
for lemma in lemmata_by_length[2]:
    print(lemma)



Two-word lemmata:
Av. Qt.
abar abāgēnīdan
abar burdan
abar būdan
abar dādan
abar dāštan
abar dīdan
abar griftan
abar guftan
abar gumēxtan
abar hangēzēnīdan
abar hištan
abar kirdan
abar kirrēnīdan
abar madan
abar māndan
abar nigerīdan
abar nihādan
abar pad
abar padīriftan
abar pahikārdan
abar paydāgīhistan
abar pādan
abar raftan
abar rasīdan
abar sazīdan
abar spixtan
abar stadan
abar waštan
abar xwandan
abar xāstan
abar āmadan
abar āwištan
abar āwurdan
abar ēstādan
abaxš būdan
abaxšišn kirdan
abestām kirdan
aboxšānišn kirdan
abzār frazendxwāhišnīh
abzōn kirdan
abzūd xwadāy
abāg burdan
abāg būdan
abāg rasīdan
abāg xuftan
abāg āwurdan
abāz abespārdan
abāz abgandan
abāz būdan
abāz dādan
abāz dāštan
abāz griftan
abāz guftan
abāz hambārīdan
abāz kirdan
abāz nimūdan
abāz nišāstan
abāz padīriftan
abāz pahikōftan
abāz raftan
abāz rasīdan
abāz stadan
abāz stūdan
abāz tōxtan
abāz waštan
abāz widardan
abāz windādan
abāz wirāstan
abāz wirāyīhistan
abāz wistan
abāz wēxtan
abāz āmadan
abāz āwurdan
a

In [ ]:
import csv
from functools import lru_cache
from collections import defaultdict

file_path = r"MPCD\dictionary exports\mpcd_dictionary-jan2026.tsv"

EXCLUDED_P1_SPLIT_LEMMATA = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy", "agīh"
}

# --------------------------------------------------
# Step 1: read lemmata
# --------------------------------------------------
lemmata = []

with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        lemma = row["word"].strip()
        if lemma:
            lemmata.append(lemma)

lemma_set = set(lemmata)

# --------------------------------------------------
# Step 2: meaningful atomic lemmata
# --------------------------------------------------
atoms = {
    l for l in lemma_set
    if " " not in l
    and len(l) >= 3
    and l not in EXCLUDED_P1_SPLIT_LEMMATA
}

# --------------------------------------------------
# Step 3: recursive segmentation
# --------------------------------------------------
@lru_cache(maxsize=None)
def segment(word):
    """
    Returns a list of valid segmentations.
    Each segmentation is a tuple of atomic lemmata.
    """
    results = []

    for i in range(3, len(word) + 1):
        prefix = word[:i]

        if prefix in atoms:
            if i == len(word):
                results.append((prefix,))
            else:
                for rest in segment(word[i:]):
                    results.append((prefix,) + rest)

    return results

# --------------------------------------------------
# Step 4: analyze one-word lemmata
# --------------------------------------------------
compound_stats = defaultdict(int)
compound_examples = defaultdict(list)

for lemma in atoms:
    decompositions = segment(lemma)

    # remove trivial decomposition (lemma itself)
    decompositions = [d for d in decompositions if len(d) >= 2]

    if decompositions:
        # choose the decomposition with the most parts
        best = max(decompositions, key=len)
        n = len(best)

        compound_stats[n] += 1
        compound_examples[n].append((lemma, best))

# --------------------------------------------------
# Step 5: reporting
# --------------------------------------------------
print("One-word compound lemmata (fully decomposable):\n")

for n in sorted(compound_stats):
    if n <= 3:
        label = f"{n}-part compounds"
        print(f"{label}: {compound_stats[n]}")
    else:
        print(f"{n}-part compounds: {compound_stats[n]}")

four_plus = sum(v for k, v in compound_stats.items() if k >= 4)
print(f"\nFour or more parts (total): {four_plus}")


One-word compound lemmata (fully decomposable):

2-part compounds: 4860
3-part compounds: 294
4-part compounds: 20
5-part compounds: 2
6-part compounds: 2
7-part compounds: 1
8-part compounds: 1

Four or more parts (total): 26


In [12]:
#inspect examples
print("\nExamples of three-part compounds:")
for l in compound_examples[3][:200]:
    print(l)



Examples of three-part compounds:
('nōgnāwarǰadag', ('nōg', 'nāwar', 'ǰadag'))
('frāznazdtomīh', ('frāz', 'nazd', 'tomīh'))
('dādgušnasp', ('dād', 'gušn', 'asp'))
('mehmānpadišīh', ('meh', 'mān', 'padišīh'))
('dagrandzanišn', ('dagr', 'and', 'zanišn'))
('andarwāyzamīg', ('andar', 'wāy', 'zamīg'))
('wispfrazānagīhpēsīd', ('wisp', 'frazānagīh', 'pēsīd'))
('xwēštanmenīh', ('xwēš', 'tan', 'menīh'))
('frāydādārgēhān', ('frāy', 'dādār', 'gēhān'))
('māywāygōwišnīh', ('māy', 'wāy', 'gōwišnīh'))
('mayānagmenīdār', ('may', 'ānag', 'menīdār'))
('dušpāddāšnīh', ('duš', 'pād', 'dāšnīh'))
('gōšōsrūdxradēwēnag', ('gōšōsrūd', 'xrad', 'ēwēnag'))
('ǰuddēwdād', ('ǰud', 'dēw', 'dād'))
('kustagtarnēmag', ('kustag', 'tar', 'nēmag'))
('pēšābustan', ('pēš', 'ābus', 'tan'))
('frāztompedīh', ('frāz', 'tom', 'pedīh'))
('ātaxšpahrēzkirdan', ('ātaxš', 'pahrēz', 'kirdan'))
('dwēsadhaftādhašt', ('dwēsad', 'haftād', 'hašt'))
('frāztomwaxšišnīh', ('frāz', 'tom', 'waxšišnīh'))
('windistan', ('win', 'dis', 'tan'))
('kā

In [ ]:
#overlapping compounds and MWE in the dictionary
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------
file_path = r"MPCD\dictionary exports\mpcd_dictionary-jan2026.tsv"
verb_csv_path = r"MPCD\dictionary exports\verb_list.csv"

# --------------------------------------------------
# DATA LOADING
# --------------------------------------------------
single_lemmata = set()
mwe_entries = set() # Words with spaces
irregular_verbs = set()

# Load Irregular Verbs
try:
    with open(verb_csv_path, encoding="utf-8") as f:
        for line in f:
            if ',' in line:
                parts = line.strip().split(',')
                inf = parts[0].strip().lower()
                pres = parts[1].strip().lower().replace('-', '')
                irregular_verbs.add(inf)
                irregular_verbs.add(pres)
except FileNotFoundError:
    pass

# Load Dictionary and separate Single words from MWEs
with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        raw_word = row["word"].strip().lower()
        if not raw_word: continue
        
        if " " in raw_word:
            mwe_entries.add(raw_word)
        else:
            single_lemmata.add(raw_word)

# All valid building blocks for a compound
all_bases = single_lemmata.union(irregular_verbs)

# --------------------------------------------------
# ANALYSIS: FIND OVERLAPS
# --------------------------------------------------

results = []

# Map collapsed MWEs to their original spaced versions for easy lookup
# e.g. {"rōzxwar": "rōz xwar"}
collapsed_mwe_map = {m.replace(" ", ""): m for m in mwe_entries}

print(f"Analyzing {len(single_lemmata)} single words and {len(mwe_entries)} MWEs...\n")

for word in single_lemmata:
    # 1. Is this word a potential compound? 
    # (Check every split point)
    for i in range(2, len(word) - 1):
        part1 = word[:i]
        part2 = word[i:]
        
        if part1 in all_bases and part2 in all_bases:
            # We found a compound! 
            # 2. Does this specific compound exist as an MWE?
            if word in collapsed_mwe_map:
                original_mwe = collapsed_mwe_map[word]
                
                # Check if the MWE split matches our compound split
                # (e.g. word "rōzxwar" split into "rōz"+"xwar" matches MWE "rōz xwar")
                mwe_parts = original_mwe.split(" ")
                if part1 == mwe_parts[0] and part2 == mwe_parts[1]:
                    results.append({
                        "combined": word,
                        "split_compound": f"{part1} + {part2}",
                        "mwe": original_mwe
                    })
                    break # Avoid duplicate splits for the same word

# --------------------------------------------------
# OUTPUT
# --------------------------------------------------

print("="*70)
print(f"{'COMPOUND (Single Word)':<25} | {'MWE (Several Words)':<25}")
print("-" * 70)

if not results:
    print("No overlaps found.")
else:
    for res in results:
        print(f"{res['combined']:<25} | {res['mwe']:<25}")

print("-" * 70)
print(f"Total Overlaps Found: {len(results)}")

# Optional: Detailed view
if len(results) > 0:
    print("\nDetailed Analysis:")
    for res in results:
        print(f"The concept '{res['combined']}' exists as both a compound ({res['split_compound']})")
        print(f"and as a Multi-Word Expression ('{res['mwe']}').")
        print("-" * 20)

Analyzing 18012 single words and 1323 MWEs...

COMPOUND (Single Word)    | MWE (Several Words)      
----------------------------------------------------------------------
abāzgriftan               | abāz griftan             
abāzwaštan                | abāz waštan              
xwadāykāmag               | xwadāy kāmag             
wehdēn                    | weh dēn                  
andargumēxtan             | andar gumēxtan           
bērustan                  | bē rustan                
andarburdan               | andar burdan             
bowandagmenīdan           | bowandag menīdan         
wanykirdan                | wany kirdan              
frōdmurdan                | frōd murdan              
wehmenišn                 | weh menišn               
nigānkirdan               | nigān kirdan             
rēmankirdan               | rēman kirdan             
tarmenīdan                | tar menīdan              
haftkišwar                | haft kišwar              
abāzstūdan        

In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

file_path = r"MPCD\dictionary exports\mpcd_dictionary-jan2026.tsv"

EXCLUDED_P1_SPLIT_LEMMATA = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"
}
EXCLUDED_P2_SUFFIX_LEMMATA = {
    "čand", "dard", "mard", "sad", "ohrmazy", "rōd",
    "fraward", "wimand", "pxrad", "xrafstar",
    "andar", "aštar", "pahikār"
}


suffix_groups = [
    ["gārīh", "kārīh", "dārīh", "tārīh", "karīh", "garīh"],
    ["dar", "tar"],
    ["tom", "dom"],
    ["āwand", "awend"],
    ["karīgīh", "kārīgīh", "gārīgīh", "garīgīh"],
    ["wandīh", "wendīh", "āwandīh"],
    ["pad", "bad", "bed"],
    ["padīh", "badīh", "bedīh"],
    ["dār", "tār"],
    ["kārīg", "gārīg"],
    ["tan", "dan"],
    ["išnīh"],
    ["išn"]
]

# --------------------------------------------------
# STEP 1: READ ALL LEMMATA
# --------------------------------------------------

lemmata = []

with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        lemma = row["word"].strip()
        if lemma:
            lemmata.append(lemma)

lemma_set = set(lemmata)

# --------------------------------------------------
# STEP 2: FILTER MEANINGFUL ONE-WORD LEMMATA
# --------------------------------------------------

atoms = {
    l for l in lemma_set
    if " " not in l
    and len(l) >= 3
    and l not in EXCLUDED_P1_SPLIT_LEMMATA
    and l not in EXCLUDED_P2_SUFFIX_LEMMATA
}

print(f"Total distinct one-word lemmata: {len(atoms)}")

# --------------------------------------------------
# STEP 3: COUNT SUFFIX DERIVATIVES
# --------------------------------------------------

group_counts = []
suffix_counts = defaultdict(int)

for group in suffix_groups:
    group_matches = set()

    for lemma in atoms:
        for suffix in group:
            if lemma.endswith(suffix) and len(lemma) > len(suffix):
                group_matches.add(lemma)
                suffix_counts[suffix] += 1

    group_counts.append((group, len(group_matches)))

# --------------------------------------------------
# STEP 4: REPORT RESULTS
# --------------------------------------------------

print("\nDerivatives per suffix group (combined):\n")

for group, count in group_counts:
    print(f"{', '.join(group)} → {count}")

print("\nDerivatives per individual suffix:\n")

for suffix in sorted(suffix_counts):
    print(f"{suffix}: {suffix_counts[suffix]}")


Total distinct one-word lemmata: 17939

Derivatives per suffix group (combined):

gārīh, kārīh, dārīh, tārīh, karīh, garīh → 627
dar, tar → 77
tom, dom → 32
āwand, awend → 17
karīgīh, kārīgīh, gārīgīh, garīgīh → 4
wandīh, wendīh, āwandīh → 32
pad, bad, bed → 26
padīh, badīh, bedīh → 15
dār, tār → 631
kārīg, gārīg → 8
tan, dan → 1340
išnīh → 1311
išn → 877

Derivatives per individual suffix:

bad: 2
badīh: 1
bed: 22
bedīh: 14
dan: 737
dar: 26
dom: 7
dār: 454
dārīh: 357
garīgīh: 1
garīh: 61
gārīg: 2
gārīh: 16
išn: 877
išnīh: 1311
karīh: 11
kārīg: 6
kārīgīh: 3
kārīh: 77
pad: 2
tan: 603
tar: 51
tom: 25
tār: 177
tārīh: 105
wandīh: 31
wendīh: 1
āwand: 17
āwandīh: 14


In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

EXCLUDED_P1_SPLIT_LEMMATA = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"
}

EXCLUDED_P2_SUFFIX_LEMMATA = {
    "čand", "dard", "mard", "sad", "ohrmazd", "rōd",
    "fraward", "wimand", "pxrad", "xrafstar",
    "andar", "aštar", "pahikār"
}

# Define suffix chains (each chain is a list of ordered suffixes, left-to-right)
suffix_chains = {
    "garīh": ["gar", "īh"],
    "karīh": ["kar", "īh"],
    "gārīh": ["gār", "īh"],
    "kārīh": ["kār", "īh"],
    "dārīh": ["dār", "īh"],
    "tārīh": ["tār", "īh"],
    "bārīh": ["bār", "īh"],
    "karīgīh": ["karīg", "īh"],
    "karīgīh": ["kar", "īg", "īh"],
    "kārīgīh": ["kārīg", "īh"],
    "kārīgīh": ["kār", "īg", "īh"],
    "garīgīh": ["karīg", "īh"],
    "garīgīh": ["kar", "īg", "īh"],
    "gārīgīh": ["gārīg", "īh"],
    "gārīgīh": ["gār", "īg", "īh"],
    "wandīh": ["wand", "īh"],
    "padīh": ["pad", "īh"],
    "išnīh": ["išn", "īh"],
    "kārīg": ["kār", "īg"],
    "karīg": ["kar", "īg"],
    "gārīg": ["gār", "īg"],
    "garīg": ["gar", "īg"],
    "bedīh": ["bed", "īh"],
    "badīh": ["bad", "īh"],
    "tarīh": ["tār", "īh"],
    "darīh": ["dār", "īh"],
    "tārīh": ["tar", "īh"],
    "dārīh": ["dar", "īh"],
}

# --------------------------------------------------
# STEP 1: READ ALL LEMMATA
# --------------------------------------------------

lemmata = []
with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        lemma = row["word"].strip()
        if lemma:
            lemmata.append(lemma)

lemma_set = set(lemmata)

# --------------------------------------------------
# STEP 2: FILTER MEANINGFUL ONE-WORD LEMMATA
# --------------------------------------------------

atoms = {
    l for l in lemma_set
    if " " not in l
    and len(l) >= 3
    and l not in EXCLUDED_P1_SPLIT_LEMMATA
    and l not in EXCLUDED_P2_SUFFIX_LEMMATA
}

print(f"Total distict one-word lemmata: {len(atoms)}")

# --------------------------------------------------
# STEP 3: DECOMPOSE LEMMATA AND ANALYZE SUFFIX CHAINS
# --------------------------------------------------

# Stats per chain
chain_stats = defaultdict(lambda: {"total": 0, "stem_derived": 0, "derived_intermediate": 0})
# Optional: keep detailed intermediate info
chain_examples = defaultdict(list)

for lemma in atoms:
    for chain_name, chain_suffixes in suffix_chains.items():
        # Check if lemma ends with full chain
        full_suffix = "".join(chain_suffixes)
        if lemma.endswith(full_suffix) and len(lemma) > len(full_suffix):
            base = lemma
            intermediates = []
            # Peel suffixes right-to-left
            for suffix in reversed(chain_suffixes):
                if base.endswith(suffix):
                    base = base[:-len(suffix)]
                    intermediates.append(base)
            intermediates.reverse()  # left-to-right order

            # Check attestation of each intermediate base
            intermediate_attested = [b in atoms for b in intermediates]

            # Classify
            if all(intermediate_attested):
                chain_stats[chain_name]["stem_derived"] += 1
            else:
                chain_stats[chain_name]["derived_intermediate"] += 1

            chain_stats[chain_name]["total"] += 1
            # store example
            chain_examples[chain_name].append({
                "lemma": lemma,
                "base": base,
                "intermediates": intermediates,
                "attested": intermediate_attested
            })

# --------------------------------------------------
# STEP 4: REPORT RESULTS
# --------------------------------------------------

print("\nSuffix chain decomposition stats:\n")
for chain_name, stats in chain_stats.items():
    print(f"Chain '{chain_name}':")
    print(f"  Total derivatives: {stats['total']}")
    print(f"  Derived directly from stem (all intermediates attested): {stats['stem_derived']}")
    print(f"  Derived from intermediate base(s) missing: {stats['derived_intermediate']}\n")

# Optional: print some examples
print("\nExamples of decomposition:\n")
for chain_name, examples in chain_examples.items():
    print(f"Chain '{chain_name}' (showing up to 5 examples):")
    for ex in examples[:5]:
        print(f"  Lemma: {ex['lemma']}")
        print(f"    Intermediates: {ex['intermediates']}")
        print(f"    Attestation: {ex['attested']}")
    print()


Total distict one-word lemmata: 17938

Suffix chain decomposition stats:

Chain 'darīh':
  Total derivatives: 357
  Derived directly from stem (all intermediates attested): 9
  Derived from intermediate base(s) missing: 348

Chain 'išnīh':
  Total derivatives: 1311
  Derived directly from stem (all intermediates attested): 52
  Derived from intermediate base(s) missing: 1259

Chain 'dārīh':
  Total derivatives: 12
  Derived directly from stem (all intermediates attested): 0
  Derived from intermediate base(s) missing: 12

Chain 'tarīh':
  Total derivatives: 105
  Derived directly from stem (all intermediates attested): 4
  Derived from intermediate base(s) missing: 101

Chain 'kārīh':
  Total derivatives: 77
  Derived directly from stem (all intermediates attested): 9
  Derived from intermediate base(s) missing: 68

Chain 'wandīh':
  Total derivatives: 31
  Derived directly from stem (all intermediates attested): 1
  Derived from intermediate base(s) missing: 30

Chain 'bedīh':
  Total

"""
# MORPHOLOGICAL PATTERN INDUCTION & GHOST STEM DETECTOR

The script below performs unsupervised morphological discovery on a dictionary dataset. 
Its goal is to identify "new" (previously undefined) suffixes and identify 
"ghost stems"—word bases that never appear alone in the dictionary but 
consistently appear with specific suffixes.

### HOW IT WORKS
The program follows a four-stage pipeline:

1. AFFIX STRIPPING (De-noising):
   To find new patterns, the script must first "blind" itself to known ones. 
   It recursively strips a predefined list of known prefixes (e.g., 'ni-', 'fra-') 
   and known suffixes (e.g., '-īh', '-īg'). The remaining part of the word 
   is referred to as the 'Residue'.

2. RESIDUE ANALYSIS (Suffix Induction):
   The script examines the endings of all 'Residues'. If a specific string of 
   characters (like '-ōmand' or '-ag') appears at the end of many different 
   residues, it is flagged as a 'Candidate Suffix'.

3. LINGUISTIC CONSTRAINTS (CV Validation):
   To prevent the machine from suggesting meaningless character fragments, 
   a "CV constraint" is applied. A candidate suffix is only valid if the 
   remaining stem contains at least one Vowel and one Consonant. This ensures 
   the stem remains a phonologically plausible Middle Persian root.

4. GHOST STEM IDENTIFICATION:
   The script performs a "cross-reference" check. For every word containing 
   a newly discovered suffix, it checks if the 'Stem' exists as an independent 
   entry in the dictionary. 
   - If the stem exists: It's a standard derivation.
   - If the stem is missing: It identifies a 'Ghost Stem' (a bound morpheme 
     or a lexical gap in the dictionary).

INPUT REQUIREMENTS:
- A TSV file with a "word" column.
- Transcription-specific vowel definitions (for the CV constraint).

OUTPUT:
- A frequency list of top potential new suffixes.
- A list of words containing those suffixes where the base stem is unattested.
"""

In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

PREFIXES = {
    "fra", "ni", "pēš", "ham", "wi", "uz", "abar", "andar", "abāz",
    "be", "pad", "tar", "ul", "awis", "a", "an", "ō", "hamēw", "hamē", 
    "ǰud", "duš", "pād", "hām", "hu", "nē", "bē", "frā"
}

KNOWN_SUFFIXES = {
    "īh", "īg", "ān", "išn", "tar", "tom", "wand", "war", "ag", "ak", 
    "gar", "gār", "kar", "kār", "dār", "tār", "bār", "bed", "bad", "ōmand", "garīh", "gārīh", "kārīh", "dārīh", "tārīh", "karīh", "dar", "tar",
    "tom", "dom", "āwand", "awend", "karīgīh", "kārīgīh", "gārīgīh", "garīgīh",
    "wandīh", "wendīh", "āwandīh", "pad", "bad", "bed", "padīh", "badīh", "bedīh",
    "dār", "tār", "kārīg", "gārīg", "tan", "dan", "išnīh", "īhā", "āg", "ār", 
    "ēn", "gēn", "ist", "īd", "ād", "gān", "īdārīh", "īdār"
}

VOWELS = "aeiouāēīōū"

def is_cv_vc_compliant(stem):
    if len(stem) < 2: return False
    has_vowel = any(v in stem for v in VOWELS)
    has_consonant = any(c not in VOWELS for c in stem)
    return has_vowel and has_consonant

def strip_known_affixes(word):
    """Returns residue and a flag if a PREFIX was stripped."""
    current = word
    prefix_stripped = False
    changed = True
    while changed:
        original = current
        # We track if a prefix specifically is removed
        for p in PREFIXES:
            if current.startswith(p) and len(current) > len(p) + 2:
                current = current[len(p):]
                prefix_stripped = True
        for s in KNOWN_SUFFIXES:
            if current.endswith(s) and len(current) > len(s) + 2:
                current = current[:-len(s)]
        if current == original:
            changed = False
    return current, prefix_stripped

# --------------------------------------------------
# STEP 1: LOAD DATA
# --------------------------------------------------
lemmata = set()
with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        word = row["word"].strip().lower()
        if word and " " not in word:
            lemmata.add(word)

print(f"Loaded {len(lemmata)} words.")

# --------------------------------------------------
# STEP 2: DISCOVER & QUANTIFY PREFIXES
# --------------------------------------------------
stats = defaultdict(lambda: {
    "start_attested": 0, "start_ghost": 0, 
    "internal_attested": 0, "internal_ghost": 0
})
residue_info = {}

for word in lemmata:
    residue, pref_stripped = strip_known_affixes(word)
    residue_info[word] = (residue, pref_stripped)
    
    if len(residue) > 4:
        # Check start of residue for 2 or 3 char candidates
        for length in [2, 3]:
            candidate = residue[:length]
            stem = residue[length:]
            if is_cv_vc_compliant(stem) and candidate not in PREFIXES:
                pos = "internal" if pref_stripped else "start"
                status = "attested" if stem in lemmata else "ghost"
                stats[candidate][f"{pos}_{status}"] += 1

# Sort candidates
top_prefixes = sorted(
    stats.keys(), 
    key=lambda x: sum(stats[x].values()), 
    reverse=True
)[:20]

# --------------------------------------------------
# STEP 3: SUMMARY TABLE
# --------------------------------------------------
print("\n" + "="*85)
print(f"{'PREFIX CAND':<10} | {'TOTAL':<5} | {'START (Att/Gho)':<18} | {'INTERNAL (Att/Gho)':<18}")
print("-" * 85)

for c in top_prefixes:
    d = stats[c]
    total = sum(d.values())
    start_str = f"{d['start_attested']}/{d['start_ghost']}"
    int_str = f"{d['internal_attested']}/{d['internal_ghost']}"
    print(f"{c:<11} | {total:<5} | {start_str:<18} | {int_str:<18}")

# --------------------------------------------------
# STEP 4: DETAILED SAMPLES
# --------------------------------------------------
print("\n" + "="*85)
print("PHASE 2: DETAILED PREFIX EXAMPLES")
print("="*85)

for c in top_prefixes:
    d = stats[c]
    examples = {k: [] for k in d.keys()}

    for word in lemmata:
        res, pref_stripped = residue_info[word]
        if res.startswith(c):
            stem = res[len(c):]
            if is_cv_vc_compliant(stem):
                pos = "internal" if pref_stripped else "start"
                status = "attested" if stem in lemmata else "ghost"
                key = f"{pos}_{status}"
                if len(examples[key]) < 3:
                    examples[key].append(f"{word} (stem: {stem})")

    print(f"\n>>> Potential Prefix: {c}-")
    
    if d['start_attested'] > 0 or d['start_ghost'] > 0:
        print(f"    POSITION: Start of Word:")
        if d['start_attested'] > 0:
            print(f"      • Attested ({d['start_attested']}): {', '.join(examples['start_attested'])}")
        if d['start_ghost'] > 0:
            print(f"      • Ghost    ({d['start_ghost']}): {', '.join(examples['start_ghost'])}")
            
    if d['internal_attested'] > 0 or d['internal_ghost'] > 0:
        print(f"    POSITION: Internal (Follows known prefix):")
        if d['internal_attested'] > 0:
            print(f"      • Attested ({d['internal_attested']}): {', '.join(examples['internal_attested'])}")
        if d['internal_ghost'] > 0:
            print(f"      • Ghost    ({d['internal_ghost']}): {', '.join(examples['internal_ghost'])}")
    print("-" * 60)

Loaded 18012 words.

PREFIX CAND | TOTAL | START (Att/Gho)    | INTERNAL (Att/Gho)
-------------------------------------------------------------------------------------
pa          | 495   | 3/414              | 0/78              
wa          | 354   | 13/313             | 0/28              
xw          | 313   | 16/259             | 3/35              
ha          | 253   | 3/236              | 0/14              
da          | 218   | 5/142              | 8/63              
sp          | 185   | 8/120              | 24/33             
ka          | 180   | 1/164              | 0/15              
dā          | 177   | 0/138              | 1/38              
za          | 168   | 1/122              | 5/40              
pu          | 166   | 2/155              | 0/9               
dr          | 161   | 3/149              | 1/8               
ma          | 161   | 3/124              | 0/34              
pur         | 148   | 9/131              | 0/8               
st          | 145   | 14/

In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

VOWELS = "aeiouāēīōū"

# Known Prefixes 
KNOWN_PREFIXES = {
    "fra", "ni", "pēš", "ham", "wi", "uz", "abar", "andar", "abāz", 
    "be", "pad", "tar", "ul", "awis", "a", "an", "ō", "hamēw", "hamē", "nē", "bē",
    "ǰud", "duš", "pād", "hām", "hu"
}

# Known Suffixes
KNOWN_SUFFIXES = {
    "īh", "īg", "ān", "išn", "tar", "tom", "wand", "war", "ag", "ak", 
    "gar", "gār", "kar", "kār", "dār", "tār", "bār", "bed", "bad", "ōmand", 
    "garīh", "gārīh", "kārīh", "dārīh", "tārīh", "karīh", "dar", 
    "dom", "āwand", "awend", "karīgīh", "kārīgīh", "gārīgīh", "garīgīh",
    "wandīh", "wendīh", "āwandīh", "pad", "bad", "bed", "padīh", "badīh", "bedīh",
    "dār", "tār", "kārīg", "gārīg", "tan", "dan", "išnīh", "īhā", "āg", 
    "ār", "ēn", "gēn", "ist", "īd", "ād", "gān"
}

def is_cv_vc_compliant(stem):
    """Checks for at least one vowel and one consonant (length >= 2)."""
    if len(stem) < 2: return False
    has_vowel = any(v in stem for v in VOWELS)
    has_consonant = any(c not in VOWELS for c in stem)
    return has_vowel and has_consonant

def strip_known_affixes(word):
    """
    Strips known affixes and tracks if a prefix was removed.
    This helps us know if the newly discovered prefix is at the absolute 
    start of the word, or if it sits behind another prefix.
    """
    current = word
    prefix_stripped = False
    changed = True
    while changed:
        original = current
        for p in KNOWN_PREFIXES:
            if current.startswith(p) and len(current) > len(p) + 2:
                current = current[len(p):]
                prefix_stripped = True  # Flag that we removed a prefix
        for s in KNOWN_SUFFIXES:
            if current.endswith(s) and len(current) > len(s) + 2:
                current = current[:-len(s)]
        if current == original:
            changed = False
    return current, prefix_stripped

# --------------------------------------------------
# STEP 1: LOAD DATA
# --------------------------------------------------
lemmata = set()
with open(file_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        word = row["word"].strip().lower()
        if word and " " not in word:
            lemmata.add(word)

print(f"Loaded {len(lemmata)} unique words.")

# --------------------------------------------------
# STEP 2: DISCOVER & QUANTIFY
# --------------------------------------------------
# Counts: initial = at word start, internal = after a known prefix
stats = defaultdict(lambda: {
    "initial_attested": 0, "initial_ghost": 0, 
    "internal_attested": 0, "internal_ghost": 0
})
residue_info = {}

for word in lemmata:
    residue, p_stripped = strip_known_affixes(word)
    residue_info[word] = (residue, p_stripped)
    
    # Check the START of the residue
    if len(residue) > 4:
        for length in [2, 3]:
            candidate = residue[:length]
            stem = residue[length:]
            if is_cv_vc_compliant(stem) and candidate not in KNOWN_PREFIXES:
                pos = "internal" if p_stripped else "initial"
                status = "attested" if stem in lemmata else "ghost"
                stats[candidate][f"{pos}_{status}"] += 1

# Sort candidates by total occurrences (top 20)
top_candidates = sorted(
    [c for c in stats.keys() if sum(stats[c].values()) >= 5], 
    key=lambda x: sum(stats[x].values()), 
    reverse=True
)[:20]

# --------------------------------------------------
# STEP 3: SUMMARY TABLE
# --------------------------------------------------
print("\n" + "="*85)
print(f"{'CANDIDATE':<10} | {'TOTAL':<5} | {'START (Att/Gho)':<18} | {'INTERNAL (Att/Gho)':<18}")
print("-" * 85)

for c in top_candidates:
    d = stats[c]
    total = sum(d.values())
    start_str = f"{d['initial_attested']}/{d['initial_ghost']}"
    int_str = f"{d['internal_attested']}/{d['internal_ghost']}"
    print(f"{c+'-':<10} | {total:<5} | {start_str:<18} | {int_str:<18}")

# --------------------------------------------------
# STEP 4: DETAILED SAMPLES
# --------------------------------------------------
print("\n" + "="*85)
print("PHASE 2: DETAILED EXAMPLES")
print("="*85)

for c in top_candidates:
    d = stats[c]
    examples = {k: [] for k in d.keys()}

    for word in lemmata:
        res, p_stripped = residue_info[word]
        if res.startswith(c):
            stem = res[len(c):]
            if is_cv_vc_compliant(stem):
                pos = "internal" if p_stripped else "initial"
                status = "attested" if stem in lemmata else "ghost"
                key = f"{pos}_{status}"
                if len(examples[key]) < 3:
                    examples[key].append(f"{word} (from {c}-{stem})")

    print(f"\n>>> Pattern: {c}-")
    
    if d['initial_attested'] > 0 or d['initial_ghost'] > 0:
        print(f"    PREFIX POSITION (Start of Word):")
        if d['initial_attested'] > 0:
            print(f"      • Attested ({d['initial_attested']}): {', '.join(examples['initial_attested'])}")
        if d['initial_ghost'] > 0:
            print(f"      • Ghost    ({d['initial_ghost']}): {', '.join(examples['initial_ghost'])}")
            
    if d['internal_attested'] > 0 or d['internal_ghost'] > 0:
        print(f"    INFIX POSITION (After known Prefix):")
        if d['internal_attested'] > 0:
            print(f"      • Attested ({d['internal_attested']}): {', '.join(examples['internal_attested'])}")
        if d['internal_ghost'] > 0:
            print(f"      • Ghost    ({d['internal_ghost']}): {', '.join(examples['internal_ghost'])}")
    print("-" * 60)

Loaded 18012 unique words.

CANDIDATE  | TOTAL | START (Att/Gho)    | INTERNAL (Att/Gho)
-------------------------------------------------------------------------------------
pa-        | 495   | 3/414              | 0/78              
fr-        | 369   | 7/333              | 1/28              
wa-        | 354   | 13/313             | 0/28              
xw-        | 309   | 16/259             | 3/31              
frā-       | 281   | 8/251              | 1/21              
ha-        | 253   | 3/236              | 0/14              
da-        | 217   | 5/142              | 8/62              
sp-        | 185   | 8/120              | 24/33             
ka-        | 180   | 1/164              | 0/15              
dā-        | 177   | 0/138              | 1/38              
pu-        | 166   | 2/155              | 0/9               
dr-        | 161   | 3/149              | 1/8               
ma-        | 161   | 3/124              | 0/34              
za-        | 160   | 1/122      

In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------
verb_csv_path = r"\MPCD\dictionary exports\verb_list.csv"

PREFIXES = {
    "fra", "ni", "pēš", "ham", "wi", "uz", "abar", "andar", "abāz",
    "be", "pad", "tar", "ul", "awis", "a", "an", "ō", "hamēw", "hamē", 
    "ǰud", "duš", "pād", "hām", "hu", "nē", "bē"
}

KNOWN_SUFFIXES = {
    "īh", "īg", "ān", "išn", "tar", "tom", "wand", "war", "ag", "ak", 
    "gar", "gār", "kar", "kār", "dār", "tār", "bār", "bed", "bad", "ōmand", 
    "garīh", "gārīh", "kārīh", "dārīh", "tārīh", "karīh", "dar", "tar",
    "tom", "dom", "āwand", "awend", "karīgīh", "kārīgīh", "gārīgīh", "gerīgīh",
    "wandīh", "wendīh", "āwandīh", "pad", "bad", "bed", "padīh", "badīh", "bedīh",
    "dār", "tār", "kārīg", "gārīg", "tan", "dan", "išnīh", "īhā", "āg", "ār", 
    "ēn", "gēn", "ist", "īd", "ād", "gān", "īdārīh", "īdār", "ēnīdan", "īdan", 
    "dan", "tan", "īhistan", "kird", "gird" 
}

EXCLUDED_CORES = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"
}

VOWELS = "aeiouāēīōū"

# --------------------------------------------------
# DATA LOADING
# --------------------------------------------------
irregular_present_map = {}
lemmata = set()

def load_irregular_verbs(path):
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                if ',' in line:
                    parts = line.strip().split(',')
                    inf, pres = parts[0].strip().lower(), parts[1].strip().lower()
                    if pres.endswith('-'): pres = pres[:-1]
                    irregular_present_map[pres] = inf
    except FileNotFoundError: pass

def load_dictionary(path):
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            word = row["word"].strip().lower()
            if word and " " not in word: lemmata.add(word)

# --------------------------------------------------
# LOGIC FUNCTIONS
# --------------------------------------------------

def is_valid_middle_persian_stem(stem):
    if len(stem) < 2 or stem in EXCLUDED_CORES: return False
    has_v = any(v in stem for v in VOWELS)
    has_c = any(c not in VOWELS for c in stem)
    return has_v and has_c and stem[-1] not in VOWELS

def get_verb_link(stem, lemmata_set):
    if stem in irregular_present_map: return irregular_present_map[stem]
    if stem.endswith("īhist"):
        inf = stem + "an"
        if inf in lemmata_set: return inf
    for ending in ["īdan", "dan", "tan", "an"]:
        inf = stem + ending
        if inf in lemmata_set: return inf
    return None

def strip_known_affixes(word):
    current = word
    suffix_stripped = False
    changed = True
    sort_p = sorted(list(PREFIXES), key=len, reverse=True)
    sort_s = sorted(list(KNOWN_SUFFIXES), key=len, reverse=True)
    while changed:
        original = current
        for p in sort_p:
            if current.startswith(p):
                rem = current[len(p):]
                if is_valid_middle_persian_stem(rem):
                    current = rem; break
        for s in sort_s:
            if current.endswith(s):
                rem = current[:-len(s)]
                if is_valid_middle_persian_stem(rem):
                    current = rem; suffix_stripped = True; break
        if current == original: changed = False
    return current, suffix_stripped

# --------------------------------------------------
# EXECUTION
# --------------------------------------------------
load_irregular_verbs(verb_csv_path)
load_dictionary(dictionary_path)

global_exclusion = set().union(KNOWN_SUFFIXES, lemmata, irregular_present_map.keys(), irregular_present_map.values())

stats = defaultdict(lambda: {"final_attested": 0, "final_ghost": 0, "internal_attested": 0, "internal_ghost": 0})
residue_info = {}

for word in lemmata:
    residue, stripped = strip_known_affixes(word)
    residue_info[word] = (residue, stripped)
    
    if len(residue) > 4:
        for length in range(6, 1, -1): 
            if len(residue) <= length + 1: continue
            candidate = residue[-length:]
            stem = residue[:-length]
            
            if is_valid_middle_persian_stem(stem) and candidate not in global_exclusion:
                
                # --- OVERLAP PROTECTION ---
                start_idx = word.find(residue)
                if start_idx != -1:
                    end_of_residue_idx = start_idx + len(residue)
                    if end_of_residue_idx < len(word):
                        next_char = word[end_of_residue_idx]
                        if (candidate + next_char) in KNOWN_SUFFIXES:
                            continue 

                pos = "internal" if stripped else "final"
                v_match = get_verb_link(stem, lemmata)
                status = "attested" if (stem in lemmata or v_match) else "ghost"
                stats[candidate][f"{pos}_{status}"] += 1
                break

# --------------------------------------------------
# OUTPUT
# --------------------------------------------------
top_candidates = sorted(stats.keys(), key=lambda x: sum(stats[x].values()), reverse=True)[:25]

print("\n" + "="*85)
print("SUFFIX DISCOVERY: WITH OVERLAP PROTECTION & EXCLUSIONS")
print("="*85)
print(f"{'CANDIDATE':<10} | {'TOTAL':<5} | {'END (Att/Gho)':<18} | {'INTERNAL (Att/Gho)':<18}")
print("-" * 85)

for c in top_candidates:
    d = stats[c]
    total = sum(d.values())
    
    # CALCULATE DISPLAY STRINGS BEFORE PRINTING (Prevents SyntaxError)
    end_display = f"{d['final_attested']}/{d['final_ghost']}"
    int_display = f"{d['internal_attested']}/{d['internal_ghost']}"
    
    print(f"-{c:<9} | {total:<5} | {end_display:<18} | {int_display:<18}")


# Examples Phase
print("\n" + "="*85)
print("PHASE 2: DETAILED EXAMPLES")
print("="*85)

for c in top_candidates:
    d = stats[c]
    examples = {k: [] for k in d.keys()}
    for word in lemmata:
        res, stripped = residue_info[word]
        found_c = None
        for length in range(6, 1, -1):
            if len(res) > length + 1:
                cand = res[-length:]
                if is_valid_middle_persian_stem(res[:-length]) and cand not in global_exclusion:
                    found_c = cand; break
        if found_c == c:
            stem = res[:-len(c)]
            verb_link = get_verb_link(stem, lemmata)
            pos = "internal" if stripped else "final"
            status = "attested" if (stem in lemmata or verb_link) else "ghost"
            key = f"{pos}_{status}"
            if len(examples[key]) < 3:
                display_stem = f"{stem} (*{verb_link})" if verb_link and stem not in lemmata else stem
                examples[key].append(f"{word} (stem: {display_stem})")

    print(f"\n>>> Candidate: -{c}")
    if d['final_attested'] + d['final_ghost'] > 0:
        ex_list = examples['final_attested'] + examples['final_ghost']
        print(f"    SUFFIX: Att: {d['final_attested']} | Gho: {d['final_ghost']} | Ex: {', '.join(ex_list)}")
    if d['internal_attested'] + d['internal_ghost'] > 0:
        ex_list = examples['internal_attested'] + examples['internal_ghost']
        print(f"    INFIX:  Att: {d['internal_attested']} | Gho: {d['internal_ghost']} | Ex: {', '.join(ex_list)}")


SUFFIX DISCOVERY: WITH OVERLAP PROTECTION & EXCLUSIONS
CANDIDATE  | TOTAL | END (Att/Gho)      | INTERNAL (Att/Gho)
-------------------------------------------------------------------------------------
-an        | 196   | 85/39              | 43/29             
-ar        | 129   | 10/46              | 12/61             
-ām        | 98    | 6/19               | 10/63             
-āh        | 90    | 5/26               | 14/45             
-ang       | 76    | 6/36               | 11/23             
-ad        | 72    | 3/34               | 1/34              
-raw       | 65    | 0/3                | 32/30             
-ah        | 57    | 0/11               | 5/41              
-ūd        | 55    | 0/25               | 2/28              
-men       | 50    | 0/1                | 32/17             
-est       | 45    | 0/2                | 34/9              
-išt       | 42    | 10/8               | 12/12             
-ēz        | 34    | 0/2                | 7/25              
-ōš 

In [ ]:
#inspect examples
print("\nExamples of three-part compounds:")
for l in compound_examples[3][:200]:
    print(l)



Examples of three-part compounds:
('nōgnāwarǰadag', ('nōg', 'nāwar', 'ǰadag'))
('frāznazdtomīh', ('frāz', 'nazd', 'tomīh'))
('dādgušnasp', ('dād', 'gušn', 'asp'))
('mehmānpadišīh', ('meh', 'mān', 'padišīh'))
('dagrandzanišn', ('dagr', 'and', 'zanišn'))
('andarwāyzamīg', ('andar', 'wāy', 'zamīg'))
('wispfrazānagīhpēsīd', ('wisp', 'frazānagīh', 'pēsīd'))
('xwēštanmenīh', ('xwēš', 'tan', 'menīh'))
('frāydādārgēhān', ('frāy', 'dādār', 'gēhān'))
('māywāygōwišnīh', ('māy', 'wāy', 'gōwišnīh'))
('mayānagmenīdār', ('may', 'ānag', 'menīdār'))
('dušpāddāšnīh', ('duš', 'pād', 'dāšnīh'))
('gōšōsrūdxradēwēnag', ('gōšōsrūd', 'xrad', 'ēwēnag'))
('ǰuddēwdād', ('ǰud', 'dēw', 'dād'))
('kustagtarnēmag', ('kustag', 'tar', 'nēmag'))
('pēšābustan', ('pēš', 'ābus', 'tan'))
('frāztompedīh', ('frāz', 'tom', 'pedīh'))
('ātaxšpahrēzkirdan', ('ātaxš', 'pahrēz', 'kirdan'))
('dwēsadhaftādhašt', ('dwēsad', 'haftād', 'hašt'))
('frāztomwaxšišnīh', ('frāz', 'tom', 'waxšišnīh'))
('windistan', ('win', 'dis', 'tan'))
('kā

In [ ]:
import csv
from collections import defaultdict

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

PREFIXES = {
    "fra", "ni", "pēš", "ham", "wi", "uz", "abar", "andar", "abāz",
    "be", "pad", "tar", "ul", "awis", "a", "an", "ō", "hamēw", "hamē", 
    "ǰud", "duš", "pād", "hām", "hu", "nē", "bē"
}

KNOWN_SUFFIXES = {
    "īh", "īg", "ān", "išn", "tar", "tom", "wand", "war", "ag", "ak", 
    "gar", "gār", "kar", "kār", "dār", "tār", "bār", "bed", "bad", "ōmand", 
    "garīh", "gārīh", "kārīh", "dārīh", "tārīh", "karīh", "dar", "tar",
    "tom", "dom", "āwand", "awend", "karīgīh", "kārīgīh", "gārīgīh", "gerīgīh",
    "wandīh", "wendīh", "āwandīh", "pad", "bad", "bed", "padīh", "badīh", "bedīh",
    "dār", "tār", "kārīg", "gārīg", "tan", "dan", "išnīh", "īhā", "āg", "ār", 
    "ēn", "gēn", "ist", "īd", "ād", "gān", "īdārīh", "īdār", "ēnīdan", "īdan", 
    "dan", "tan", "īhistan", "kird", "gird" 
}

EXCLUDED_CORES = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"
}

VOWELS = "aeiouāēīōū"

# --------------------------------------------------
# DATA LOADING
# --------------------------------------------------
irregular_present_map = {}
lemmata = set()

def load_irregular_verbs(path):
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                if ',' in line:
                    parts = line.strip().split(',')
                    inf, pres = parts[0].strip().lower(), parts[1].strip().lower()
                    if pres.endswith('-'): pres = pres[:-1]
                    irregular_present_map[pres] = inf
    except FileNotFoundError: pass

def load_dictionary(path):
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            word = row["word"].strip().lower()
            if word and " " not in word: lemmata.add(word)

load_irregular_verbs(verb_csv_path)
load_dictionary(dictionary_path)

all_known_bases = set().union(lemmata, irregular_present_map.keys(), irregular_present_map.values())
global_exclusion = set().union(KNOWN_SUFFIXES, all_known_bases)

# --------------------------------------------------
# LOGIC FUNCTIONS
# --------------------------------------------------

def is_known_base(s):
    if not s: return False
    if s in all_known_bases: return True
    for end in ["īdan", "dan", "tan", "an"]:
        if (s + end) in lemmata: return True
    return False

def can_be_explained_as_compound(s):
    if is_known_base(s): return True
    for i in range(2, len(s) - 1):
        if is_known_base(s[:i]) and is_known_base(s[i:]): return True
    return False

def is_valid_stem(stem):
    if len(stem) < 2 or stem in EXCLUDED_CORES: return False
    has_v = any(v in stem for v in VOWELS)
    has_c = any(c not in VOWELS for c in stem)
    return has_v and has_c and stem[-1] not in VOWELS

def strip_and_analyze(word):
    current = word
    suffix_stripped = False
    
    # Strip Prefixes
    sort_p = sorted(list(PREFIXES), key=len, reverse=True)
    changed = True
    while changed:
        original = current
        for p in sort_p:
            if current.startswith(p):
                rem = current[len(p):]
                if is_valid_stem(rem): current = rem; break
        if current == original: changed = False

    # Strip Suffixes
    sort_s = sorted(list(KNOWN_SUFFIXES), key=len, reverse=True)
    changed = True
    while changed:
        original = current
        for s in sort_s:
            if current.endswith(s):
                rem = current[:-len(s)]
                if is_valid_stem(rem): current = rem; suffix_stripped = True; break
        if current == original: changed = False

    is_explained = can_be_explained_as_compound(current)
    return current, suffix_stripped, is_explained

# --------------------------------------------------
# EXECUTION
# --------------------------------------------------
stats = defaultdict(lambda: {"final_total": 0, "internal_total": 0})
residue_info = {}

for word in lemmata:
    res, stripped, explained = strip_and_analyze(word)
    residue_info[word] = (res, stripped, explained)
    
    if not explained and len(res) > 4:
        for length in range(6, 1, -1): 
            if len(res) <= length + 1: continue
            candidate = res[-length:]
            stem = res[:-length]
            
            if is_valid_stem(stem) and candidate not in global_exclusion:
                if can_be_explained_as_compound(stem): continue
                
                # Overlap protection
                start_idx = word.find(res)
                if start_idx != -1:
                    end_idx = start_idx + len(res)
                    if end_idx < len(word) and (candidate + word[end_idx]) in KNOWN_SUFFIXES:
                        continue 

                pos = "internal" if stripped else "final"
                stats[candidate][f"{pos}_total"] += 1
                break

# --------------------------------------------------
# OUTPUT - FILTERING OUT PURE INFIXES
# --------------------------------------------------
# We only take candidates that appear at the END (final_total > 0)
top_candidates = [c for c in stats.keys() if stats[c]['final_total'] > 0]
top_candidates = sorted(top_candidates, key=lambda x: sum(stats[x].values()), reverse=True)[:25]

print("\n" + "="*85)
print("SUFFIX DISCOVERY: REAL SUFFIXES (INFIX-ONLY STRINGS EXCLUDED)")
print("="*85)
for c in top_candidates:
    d = stats[c]
    total = d['final_total'] + d['internal_total']
    print(f"-{c:<11} | Total: {total:<5} | End (Suffix): {d['final_total']:<5} | Mid (Infix): {d['internal_total']}")

print("\nPHASE 2: DETAILED EXAMPLES")
for c in top_candidates:
    examples = {"final": [], "internal": []}
    for word in lemmata:
        res, stripped, explained = residue_info[word]
        if explained: continue
        
        found_c = None
        for length in range(6, 1, -1):
            if len(res) > length + 1:
                cand = res[-length:]
                stem_check = res[:-length]
                if is_valid_stem(stem_check) and cand not in global_exclusion:
                    if not can_be_explained_as_compound(stem_check):
                        found_c = cand; break
        
        if found_c == c:
            stem = res[:-len(c)]
            pos = "internal" if stripped else "final"
            if len(examples[pos]) < 3:
                examples[pos].append(f"{word} (ghost: {stem})")

    if examples["final"] or examples["internal"]:
        print(f"\n>>> Candidate: -{c}")
        if examples["final"]: print(f"    SUFFIX POSITION: {', '.join(examples['final'])}")
        if examples["internal"]: print(f"    INTERNAL POSITION: {', '.join(examples['internal'])}")


SUFFIX DISCOVERY: REAL SUFFIXES (INFIX-ONLY STRINGS EXCLUDED)
-aw          | Total: 69    | End (Suffix): 1     | Mid (Infix): 68
-an          | Total: 55    | End (Suffix): 40    | Mid (Infix): 15
-ar          | Total: 33    | End (Suffix): 6     | Mid (Infix): 27
-āh          | Total: 31    | End (Suffix): 7     | Mid (Infix): 24
-ām          | Total: 25    | End (Suffix): 4     | Mid (Infix): 21
-ah          | Total: 22    | End (Suffix): 2     | Mid (Infix): 20
-am          | Total: 20    | End (Suffix): 1     | Mid (Infix): 19
-šāy         | Total: 16    | End (Suffix): 3     | Mid (Infix): 13
-īr          | Total: 14    | End (Suffix): 3     | Mid (Infix): 11
-arz         | Total: 13    | End (Suffix): 2     | Mid (Infix): 11
-ad          | Total: 12    | End (Suffix): 3     | Mid (Infix): 9
-ast         | Total: 12    | End (Suffix): 5     | Mid (Infix): 7
-ūd          | Total: 10    | End (Suffix): 6     | Mid (Infix): 4
-ās          | Total: 9     | End (Suffix): 2     | Mid 